# Terafab Decision Twin — Comparative Simulation and Executive Reporting

This notebook orchestrates a three-scenario 2026–2030 run using the public repository:

`https://github.com/KNOWDYN/terafab-decision-twin`

It validates, simulates, compares, plots, and reports:

- `multi_year_2026_2030.json` — reference case already in the repository
- `worst_case_2026_2030.json` — near-failure stress case
- `best_case_2026_2030.json` — milestone-support case

Outputs created by this notebook:

- `comparative_summary_2026_2030.csv`
- `comparative_gates_2026_2030.csv`
- `comparative_interpretation_2026_2030.csv`
- `comparative_infographic_2026_2030.png`
- `executive_strategy_brief_2026_2030.html`
- `terafab_comparative_reporting_outputs.zip`

Important boundary: these scenarios are evidence-gated model assumptions. They are not verified Terafab operating facts and do not imply Terafab endorsement, affiliation, authorization, confidential access, or employee involvement.


In [ ]:
# Install the Terafab Decision Twin package from the public GitHub repository.
# Run this cell in Google Colab.

!rm -rf /content/terafab-decision-twin
!git clone https://github.com/KNOWDYN/terafab-decision-twin.git
%cd /content/terafab-decision-twin
!pip -q install .


In [ ]:
# Imports and runtime configuration.

from pathlib import Path
from datetime import datetime, timezone
import json
import zipfile

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from matplotlib.ticker import FuncFormatter

from terafab_decision_twin.schema import load_scenario, validate_scenario
from terafab_decision_twin.engine import run_scenario

REPO_DIR = Path("/content/terafab-decision-twin")
SCENARIO_DIR = REPO_DIR / "scenarios"

SCENARIOS = {
    "Reference": SCENARIO_DIR / "multi_year_2026_2030.json",
    "Worst case": SCENARIO_DIR / "worst_case_2026_2030.json",
    "Best case": SCENARIO_DIR / "best_case_2026_2030.json",
}

MODEL_VERSION_LABEL = "terafab-decision-twin v0.3.0"
HORIZON_LABEL = "2026–2030"
RUN_TIMESTAMP_UTC = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S UTC")

print("Scenario directory:", SCENARIO_DIR)
for name, path in SCENARIOS.items():
    print(f"{name:12s} -> {path} | exists={path.exists()}")


## Upload missing scenario files, if needed

If the two new files are not yet committed to the repository, run the upload cell below and upload:

- `worst_case_2026_2030.json`
- `best_case_2026_2030.json`

If they are already present in `scenarios/`, skip the upload cell.


In [ ]:
# Optional upload helper for Google Colab.
# Skip this cell if both new scenario JSON files already exist in scenarios/.

from google.colab import files

missing = [str(path.name) for path in SCENARIOS.values() if not path.exists()]
print("Missing before upload:", missing)

if missing:
    uploaded = files.upload()
    for filename, content in uploaded.items():
        target = SCENARIO_DIR / filename
        target.write_bytes(content)
        print("Saved", target)
else:
    print("No upload needed.")


In [ ]:
# Confirm all required scenario files are available.

missing_after_upload = [str(path) for path in SCENARIOS.values() if not path.exists()]
if missing_after_upload:
    raise FileNotFoundError("Missing required scenario files: " + ", ".join(missing_after_upload))

print("All required scenario files found.")


In [ ]:
# Validate all scenarios.

validation_rows = []
loaded_scenarios = {}

for label, path in SCENARIOS.items():
    scenario = load_scenario(path)
    errors = validate_scenario(scenario)
    loaded_scenarios[label] = scenario
    validation_rows.append({
        "scenario": label,
        "scenario_id": scenario.get("metadata", {}).get("scenario_id", ""),
        "valid": len(errors) == 0,
        "error_count": len(errors),
        "errors": "; ".join(errors)
    })

validation_df = pd.DataFrame(validation_rows)
validation_df


In [ ]:
# Stop if any scenario is invalid.

invalid = validation_df[validation_df["valid"] == False]
if not invalid.empty:
    display(invalid)
    raise ValueError("One or more scenarios failed validation. Fix the JSON before simulation.")

print("All scenarios are valid.")


In [ ]:
# Run all simulations.

results = {}
for label, scenario in loaded_scenarios.items():
    results[label] = run_scenario(scenario)

print("Simulation completed.")
for label, result in results.items():
    print(f"{label:12s} | passed={result['passed']} | failed gates={sum(1 for g in result['gates'] if not g['passed'])}")


In [ ]:
# Build summary and gate dataframes.

summary_rows = []
gate_rows = []

for label, result in results.items():
    s = result["summary"]
    scenario = loaded_scenarios[label]
    scenario_id = scenario.get("metadata", {}).get("scenario_id", label)

    summary_rows.append({
        "scenario": label,
        "scenario_id": scenario_id,
        "average_site_load_MW": s["average_site_load_MW"],
        "firm_capacity_margin_MW": s["minimum_firm_capacity_margin_MW"],
        "heat_rejection_margin_MW": s["minimum_heat_rejection_margin_MW"],
        "water_withdrawal_margin_m3_per_day": s["minimum_water_withdrawal_margin_m3_per_day"],
        "wastewater_discharge_margin_m3_per_day": s["minimum_wastewater_discharge_margin_m3_per_day"],
        "effective_yield": s["average_effective_yield"],
        "total_modeled_cost_USD": s["total_cost_USD"],
        "total_modeled_cost_USD_bn": s["total_cost_USD"] / 1e9,
        "cost_per_good_die_USD": s["cost_per_good_die_USD"],
        "emissions_tCO2": s["emissions_tCO2"],
        "emissions_MtCO2": s["emissions_tCO2"] / 1e6,
        "public_benefit_index": s["public_benefit_index"],
        "public_burden_index": s["public_burden_index"],
        "legitimacy_margin": s["legitimacy_margin"],
        "governance_risk_index": s["maximum_governance_risk_index"],
        "recommended_action": s["recommended_phase_action"],
        "passed": result["passed"],
        "failed_gate_count": sum(1 for g in result["gates"] if not g["passed"])
    })

    for gate in result["gates"]:
        gate_rows.append({
            "scenario": label,
            "scenario_id": scenario_id,
            "gate": gate["name"],
            "passed": gate["passed"],
            "status": "PASS" if gate["passed"] else "FAIL",
            "severity": gate["severity"],
            "margin": gate["margin"],
            "message": gate["message"]
        })

summary_df = pd.DataFrame(summary_rows).set_index("scenario")
gates_df = pd.DataFrame(gate_rows)

display(summary_df)
display(gates_df)


In [ ]:
# Define board/investor interpretation strings.

def action_rationale(label, row):
    if label == "Worst case":
        return (
            "Hold or redesign is indicated because the modeled stress case breaches binding "
            "infrastructure gates before economics can be interpreted as executable."
        )
    if label == "Reference":
        return (
            "Stage and monitor is indicated because the reference case clears gates while still "
            "requiring evidence discipline, permitting confidence, and infrastructure monitoring."
        )
    if label == "Best case":
        return (
            "Advance is indicated because supportive assumptions preserve power, cooling, water, "
            "yield, governance, and legitimacy headroom."
        )
    return f"Recommended action: {row['recommended_action']}; failed gates: {int(row['failed_gate_count'])}."

interpretation_rows = []
for label, row in summary_df.iterrows():
    interpretation_rows.append({
        "scenario": label,
        "recommended_action": row["recommended_action"],
        "failed_gate_count": int(row["failed_gate_count"]),
        "rationale": action_rationale(label, row)
    })

interpretation_df = pd.DataFrame(interpretation_rows).set_index("scenario")
interpretation_df


In [ ]:
# Create one-page comparative infographic.

order = ["Worst case", "Reference", "Best case"]
plot_df = summary_df.loc[order]

gate_pivot = (
    gates_df.assign(value=gates_df["passed"].astype(int))
    .pivot(index="gate", columns="scenario", values="value")
    .loc[:, order]
)

def pct_fmt(x, pos):
    return f"{100*x:.0f}%"

def money_bn_fmt(x, pos):
    return f"${x:.1f}B"

fig = plt.figure(figsize=(14, 18), constrained_layout=True)
gs = GridSpec(5, 2, figure=fig, height_ratios=[0.85, 1.05, 1.05, 1.05, 1.35])

ax_title = fig.add_subplot(gs[0, :])
ax_title.axis("off")

title = "Terafab Decision Twin | Comparative 2026–2030 Scenario Outlook"
subtitle = (
    "Reference vs. worst-case near-failure stress vs. best-case milestone-support case. "
    "Outputs are model results from evidence-gated scenario assumptions, not verified Terafab operating facts."
)
ax_title.text(0.0, 0.92, title, fontsize=20, fontweight="bold", ha="left", va="top")
ax_title.text(0.0, 0.56, subtitle, fontsize=11, ha="left", va="top", wrap=True)

headline = []
for label in order:
    row = plot_df.loc[label]
    headline.append(
        f"{label}: {row['recommended_action']} | failed gates: {int(row['failed_gate_count'])} | "
        f"yield: {row['effective_yield']:.1%} | total modeled cost: ${row['total_modeled_cost_USD_bn']:.2f}B"
    )
ax_title.text(0.0, 0.08, "\n".join(headline), fontsize=10.5, ha="left", va="bottom")

ax1 = fig.add_subplot(gs[1, 0])
plot_df[["average_site_load_MW", "firm_capacity_margin_MW"]].rename(columns={
    "average_site_load_MW": "Average load",
    "firm_capacity_margin_MW": "Firm-capacity margin"
}).plot(kind="bar", ax=ax1)
ax1.set_title("Power demand and firm-capacity margin")
ax1.set_ylabel("MW")
ax1.tick_params(axis="x", rotation=0)
ax1.axhline(0, linewidth=1)

ax2 = fig.add_subplot(gs[1, 1])
plot_df[["heat_rejection_margin_MW", "water_withdrawal_margin_m3_per_day"]].rename(columns={
    "heat_rejection_margin_MW": "Cooling margin (MW)",
    "water_withdrawal_margin_m3_per_day": "Water margin (m3/day)"
}).plot(kind="bar", ax=ax2)
ax2.set_title("Cooling and water headroom")
ax2.tick_params(axis="x", rotation=0)
ax2.axhline(0, linewidth=1)

ax3 = fig.add_subplot(gs[2, 0])
plot_df[["effective_yield", "legitimacy_margin"]].rename(columns={
    "effective_yield": "Effective yield",
    "legitimacy_margin": "Legitimacy margin"
}).plot(kind="bar", ax=ax3)
ax3.set_title("Manufacturing and public legitimacy")
ax3.yaxis.set_major_formatter(FuncFormatter(pct_fmt))
ax3.tick_params(axis="x", rotation=0)
ax3.axhline(0, linewidth=1)

ax4 = fig.add_subplot(gs[2, 1])
plot_df[["governance_risk_index"]].rename(columns={
    "governance_risk_index": "Governance risk"
}).plot(kind="bar", ax=ax4)
ax4.set_title("Governance risk index, lower is better")
ax4.set_ylabel("Index")
ax4.tick_params(axis="x", rotation=0)
ax4.axhline(0, linewidth=1)

ax5 = fig.add_subplot(gs[3, 0])
plot_df[["total_modeled_cost_USD_bn"]].rename(columns={
    "total_modeled_cost_USD_bn": "Total modeled cost"
}).plot(kind="bar", ax=ax5)
ax5.set_title("Modeled economic exposure")
ax5.yaxis.set_major_formatter(FuncFormatter(money_bn_fmt))
ax5.tick_params(axis="x", rotation=0)
ax5.axhline(0, linewidth=1)

ax6 = fig.add_subplot(gs[3, 1])
plot_df[["emissions_MtCO2"]].rename(columns={
    "emissions_MtCO2": "Emissions"
}).plot(kind="bar", ax=ax6)
ax6.set_title("Modeled emissions burden")
ax6.set_ylabel("MtCO2")
ax6.tick_params(axis="x", rotation=0)
ax6.axhline(0, linewidth=1)

ax7 = fig.add_subplot(gs[4, 0])
ax7.set_title("Gate matrix: 1 = pass, 0 = fail")
ax7.imshow(gate_pivot.values, aspect="auto", interpolation="nearest")
ax7.set_xticks(range(len(order)))
ax7.set_xticklabels(order, rotation=0)
ax7.set_yticks(range(len(gate_pivot.index)))
ax7.set_yticklabels(gate_pivot.index)
for i in range(gate_pivot.shape[0]):
    for j in range(gate_pivot.shape[1]):
        ax7.text(j, i, int(gate_pivot.iloc[i, j]), ha="center", va="center", fontsize=9)

ax8 = fig.add_subplot(gs[4, 1])
ax8.axis("off")
takeaways = ["Decision takeaways", ""]
for label in order:
    row = plot_df.loc[label]
    takeaways.extend([
        f"{label}",
        f"• Action: {row['recommended_action']}",
        f"• Failed gates: {int(row['failed_gate_count'])}",
        f"• Yield: {row['effective_yield']:.1%}",
        f"• Total modeled cost: ${row['total_modeled_cost_USD_bn']:.2f}B",
        f"• Emissions: {row['emissions_MtCO2']:.2f} MtCO2",
        ""
    ])
takeaways.extend([
    "Interpretation",
    "• Worst case shows which gates become binding first.",
    "• Reference case is a staged path requiring monitoring.",
    "• Best case shows the value of infrastructure, governance, and regulatory headroom.",
])
ax8.text(0, 1, "\n".join(takeaways), ha="left", va="top", fontsize=10)

fig.savefig("comparative_infographic_2026_2030.png", dpi=220, bbox_inches="tight")
plt.show()


In [ ]:
# Generate corrected executive HTML strategy brief.

def fmt_money_bn(x):
    return f"${x/1e9:,.2f}B"

def fmt_mw(x):
    return f"{x:,.1f} MW"

def fmt_m3_day(x):
    return f"{x:,.0f} m³/day"

def fmt_pct(x):
    return f"{x:.1%}"

def status_class(passed):
    return "pass" if passed else "fail"

def scenario_metric_table():
    rows = []
    for label in ["Worst case", "Reference", "Best case"]:
        r = summary_df.loc[label]
        rows.append(f"""
        <tr>
            <td><strong>{label}</strong></td>
            <td>{r['recommended_action']}</td>
            <td>{int(r['failed_gate_count'])}</td>
            <td>{fmt_mw(r['firm_capacity_margin_MW'])}</td>
            <td>{fmt_mw(r['heat_rejection_margin_MW'])}</td>
            <td>{fmt_m3_day(r['water_withdrawal_margin_m3_per_day'])}</td>
            <td>{fmt_pct(r['effective_yield'])}</td>
            <td>{fmt_money_bn(r['total_modeled_cost_USD'])}</td>
            <td>{r['emissions_MtCO2']:.2f} MtCO₂</td>
        </tr>
        """)
    return "\n".join(rows)

def gate_matrix_html():
    chunks = []
    for label in ["Worst case", "Reference", "Best case"]:
        scenario_gates = gates_df[gates_df["scenario"] == label]
        items = []
        for _, g in scenario_gates.iterrows():
            klass = status_class(g["passed"])
            gate_name = g["gate"].replace("_", " ").title()
            items.append(f"""
            <div class="gate-item">
                <span>{gate_name}</span>
                <span class="{klass}">{g['status']}</span>
            </div>
            """)
        chunks.append(f"""
        <section class="gate-card">
            <h3>{label}</h3>
            {''.join(items)}
        </section>
        """)
    return "\n".join(chunks)

def directive_html():
    chunks = []
    for label in ["Worst case", "Reference", "Best case"]:
        row = interpretation_df.loc[label]
        chunks.append(f"""
        <div class="directive">
            <div class="directive-label">{label}: {row['recommended_action']}</div>
            <p>{row['rationale']}</p>
        </div>
        """)
    return "\n".join(chunks)

html = f"""<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>Terafab Decision Twin | Comparative Strategy Brief</title>
<style>
:root {{
    --black: #050505;
    --white: #ffffff;
    --paper: #fbfbfb;
    --line: #d9d9d9;
    --muted: #666666;
    --soft: #f4f4f4;
}}
* {{ box-sizing: border-box; }}
body {{
    margin: 0;
    padding: 40px 20px;
    background: var(--paper);
    color: var(--black);
    font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Inter, Roboto, Arial, sans-serif;
    line-height: 1.45;
}}
main {{
    max-width: 1080px;
    margin: 0 auto;
    background: var(--white);
    border: 1px solid var(--line);
    border-radius: 28px;
    padding: 44px;
    box-shadow: 0 18px 60px rgba(0,0,0,0.06);
}}
.meta {{
    font-size: 11px;
    letter-spacing: .08em;
    text-transform: uppercase;
    color: var(--muted);
    font-weight: 700;
}}
h1 {{
    margin: 12px 0 8px;
    font-size: clamp(32px, 5vw, 56px);
    line-height: 0.98;
    letter-spacing: -0.045em;
}}
.lede {{
    max-width: 830px;
    color: #333;
    font-size: 17px;
}}
.divider {{
    height: 1px;
    background: var(--black);
    margin: 34px 0;
}}
.kpis {{
    display: grid;
    grid-template-columns: repeat(3, 1fr);
    gap: 18px;
}}
.kpi {{
    border: 1px solid var(--line);
    border-radius: 20px;
    padding: 20px;
    background: #fff;
}}
.kpi-label {{
    font-size: 11px;
    text-transform: uppercase;
    letter-spacing: .08em;
    color: var(--muted);
    font-weight: 800;
}}
.kpi-value {{
    margin-top: 12px;
    font-size: 30px;
    letter-spacing: -0.03em;
}}
table {{
    width: 100%;
    border-collapse: collapse;
    font-size: 13px;
}}
th {{
    text-align: left;
    font-size: 10px;
    text-transform: uppercase;
    letter-spacing: .08em;
    color: var(--muted);
    border-bottom: 1px solid var(--black);
    padding: 10px 8px;
}}
td {{
    border-bottom: 1px solid var(--line);
    padding: 12px 8px;
}}
.gates {{
    display: grid;
    grid-template-columns: repeat(3, 1fr);
    gap: 18px;
}}
.gate-card {{
    background: var(--soft);
    border-radius: 20px;
    padding: 18px;
}}
.gate-card h3 {{
    margin: 0 0 12px;
    font-size: 15px;
}}
.gate-item {{
    display: flex;
    justify-content: space-between;
    gap: 12px;
    border-bottom: 1px solid #e0e0e0;
    padding: 6px 0;
    font-size: 12px;
}}
.pass {{ font-weight: 800; color: #111; }}
.fail {{ font-weight: 900; color: #111; text-decoration: underline; }}
.directives {{
    display: grid;
    grid-template-columns: 1fr;
    gap: 14px;
}}
.directive {{
    border-left: 3px solid var(--black);
    padding: 4px 0 4px 16px;
}}
.directive-label {{
    font-weight: 900;
    text-transform: uppercase;
    font-size: 12px;
    letter-spacing: .05em;
}}
.directive p {{
    margin: 6px 0 0;
}}
.notice {{
    margin-top: 34px;
    color: var(--muted);
    font-size: 11px;
}}
@media (max-width: 850px) {{
    main {{ padding: 26px; border-radius: 20px; }}
    .kpis {{ grid-template-columns: 1fr; }}
    .gates {{ grid-template-columns: 1fr; }}
    table {{ display: block; overflow-x: auto; }}
}}
</style>
</head>
<body>
<main>
    <div class="meta">Evidence-Gated Scenario Brief | {MODEL_VERSION_LABEL} | Generated {RUN_TIMESTAMP_UTC}</div>
    <h1>Comparative 2026–2030 Strategy Outlook</h1>
    <p class="lede">
        This brief compares a reference case, a near-failure stress case, and a milestone-support case for Terafab-scale planning.
        It is designed to show which constraints become binding first, what conditions preserve headroom, and what decision posture follows from the model gates.
    </p>

    <div class="divider"></div>

    <section class="kpis">
        <div class="kpi">
            <div class="kpi-label">Worst-case posture</div>
            <div class="kpi-value">{summary_df.loc['Worst case', 'recommended_action']}</div>
        </div>
        <div class="kpi">
            <div class="kpi-label">Reference modeled total cost</div>
            <div class="kpi-value">{fmt_money_bn(summary_df.loc['Reference', 'total_modeled_cost_USD'])}</div>
        </div>
        <div class="kpi">
            <div class="kpi-label">Best-case effective yield</div>
            <div class="kpi-value">{fmt_pct(summary_df.loc['Best case', 'effective_yield'])}</div>
        </div>
    </section>

    <div class="divider"></div>

    <section>
        <div class="meta">Scenario Comparison</div>
        <table>
            <thead>
                <tr>
                    <th>Scenario</th>
                    <th>Action</th>
                    <th>Failed gates</th>
                    <th>Power margin</th>
                    <th>Cooling margin</th>
                    <th>Water margin</th>
                    <th>Yield</th>
                    <th>Total modeled cost</th>
                    <th>Emissions</th>
                </tr>
            </thead>
            <tbody>
                {scenario_metric_table()}
            </tbody>
        </table>
    </section>

    <div class="divider"></div>

    <section>
        <div class="meta">Gate Matrix</div>
        <div class="gates">
            {gate_matrix_html()}
        </div>
    </section>

    <div class="divider"></div>

    <section>
        <div class="meta">Strategic Directives</div>
        <div class="directives">
            {directive_html()}
        </div>
    </section>

    <p class="notice">
        © KNOWDYN. All rights reserved. Terafab is owned by its official Terafab entity.
        This project and report are independent and do not claim Terafab endorsement, authorization, affiliation, employee involvement,
        confidential access, or verified operating data. Scenario assumptions are not verified Terafab operating facts.
        Restricted source documents are not redistributed.
    </p>
</main>
</body>
</html>
"""

Path("executive_strategy_brief_2026_2030.html").write_text(html)
print("Wrote executive_strategy_brief_2026_2030.html")


In [ ]:
# Save CSV files and bundle all outputs.

summary_df.to_csv("comparative_summary_2026_2030.csv")
gates_df.to_csv("comparative_gates_2026_2030.csv", index=False)
interpretation_df.to_csv("comparative_interpretation_2026_2030.csv")

with zipfile.ZipFile("terafab_comparative_reporting_outputs.zip", "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for filename in [
        "comparative_summary_2026_2030.csv",
        "comparative_gates_2026_2030.csv",
        "comparative_interpretation_2026_2030.csv",
        "comparative_infographic_2026_2030.png",
        "executive_strategy_brief_2026_2030.html",
    ]:
        zf.write(filename)

print("Created terafab_comparative_reporting_outputs.zip")


## Interpretation discipline

Use this report as a comparative scenario product, not as a factual statement about Terafab operations.

Correct language:

- “The worst-case scenario fails power/cooling/water gates.”
- “The best-case scenario passes the model gates under supportive assumptions.”
- “The reference scenario indicates stage-and-monitor under the current assumption set.”

Incorrect language:

- “Terafab will fail.”
- “Terafab has verified these loads, costs, or yields.”
- “Terafab endorsed this model.”
- “The model contains confidential Terafab operating data.”
